In [3]:
import pandas as pd
import numpy as np

In [4]:
#carga de base de datos
taller = pd.read_csv(r"C:\Python\Clases\Talleres\Actividad1\ventas_sucias_5000.csv")

In [5]:
#mostrar las primeras filas
print(taller.head())

  cliente producto  precio cantidad      pais    metodo_pago  \
0   Maria  Monitor  1326.0      NaN      peru       Efectivo   
1   Luisa   Laptop    55.0        2     chile        Tarjeta   
2  Carlos  Monitor  1203.0        9  Colombia       Efectivo   
3   Luisa  Monitor  1304.0        3      Perú  TRANSFERENCIA   
4   Luisa  Monitor   426.0        6     chile        Tarjeta   

                 fecha  
0  2024-01-01 00:00:00  
1  2024-01-01 01:00:00  
2  2024-01-01 02:00:00  
3  2024-01-01 03:00:00  
4  2024-01-01 04:00:00  


In [6]:
taller.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 5000 entries, 0 to 4999
Data columns (total 7 columns):
 #   Column       Non-Null Count  Dtype  
---  ------       --------------  -----  
 0   cliente      5000 non-null   object 
 1   producto     5000 non-null   object 
 2   precio       4950 non-null   float64
 3   cantidad     4950 non-null   object 
 4   pais         5000 non-null   object 
 5   metodo_pago  5000 non-null   object 
 6   fecha        5000 non-null   object 
dtypes: float64(1), object(6)
memory usage: 273.6+ KB


In [7]:
#resumen estadistico
print(taller.describe())

              precio
count    4950.000000
mean     5053.823434
std     63379.982009
min        10.000000
25%       531.000000
50%      1027.000000
75%      1519.000000
max    999999.000000


La base de datos tiene 7 columnas y 5000 filas en las que encontramos los datos. La columna de precio tiene datos numericos y las restantes tienen datos en formato de texto. Las columna  de "cantidad" deben ser convertidas a formato numerico para poder trabajar en ella, al  igual que la columna fecha que debe transformarse con dicho formato. Y las columnas pais y metodo_pago tienen discrepancias en sus datos que estan escritos de formas diferentes.



Limpieza de datos

In [8]:

taller["metodo_pago"].unique()

array(['Efectivo', 'Tarjeta', 'TRANSFERENCIA', 'transferencia'],
      dtype=object)

In [9]:
#corregir tipos de datos incorrectos

#convertimos la columna "cantidad" a numérica, y la columna "fecha" a datetime
taller["cantidad"] = pd.to_numeric(taller["cantidad"], errors='coerce')
taller["fecha"] = pd.to_datetime(taller["fecha"], errors='coerce')



In [10]:
#identificar los valores nulos
taller.isnull().sum()

cliente          0
producto         0
precio          50
cantidad       100
pais             0
metodo_pago      0
fecha            0
dtype: int64

In [11]:
#manejo de nulos existentes en las columnas cantidad y precio.
# Rellenar precio con la mediana del mismo producto
taller["precio"] = taller.groupby("producto")["precio"].transform(
    lambda x: x.fillna(x.median())
)

# Rellenar cantidad con la mediana del mismo producto y país
taller["cantidad"] = taller.groupby(["producto", "pais"])["cantidad"].transform(
    lambda x: x.fillna(x.median())
)

taller["cantidad"] = taller["cantidad"].fillna(taller["cantidad"].median())
taller["precio"]= taller["precio"].fillna(taller["precio"].median())


In [12]:
#verificar que ya no hay nulos
taller.isnull().sum()

cliente        0
producto       0
precio         0
cantidad       0
pais           0
metodo_pago    0
fecha          0
dtype: int64

In [13]:
#limpieza de datos en columna de texto

taller["pais"] = taller["pais"].str.lower().str.strip()
taller["metodo_pago"] = taller["metodo_pago"].str.lower().str.strip()
taller["producto"] = taller["producto"].str.lower().str.strip()

In [14]:
taller["cliente"].unique()

array(['Maria', 'Luisa', 'Carlos', 'Juan', 'Pedro', 'Ana'], dtype=object)

In [15]:
taller["producto"].unique()

array(['monitor', 'laptop', 'celular', 'mouse', 'teclado'], dtype=object)

In [16]:
#verificamos las columnas de texto
taller["metodo_pago"].unique()

array(['efectivo', 'tarjeta', 'transferencia'], dtype=object)

In [17]:
taller["pais"].unique()

array(['peru', 'chile', 'colombia', 'perú', 'col'], dtype=object)

In [18]:
#corregir errores tipográficos en la columna pais
taller["pais"] = taller["pais"].replace({
    "col":"colombia",
    "perú": "peru"
})

Verificamos los outliers en la columna precio, dado que son una cantidad considerable, hacemos uso de la mediana para no perder la informacion de esas filas y clientes.

In [19]:
#manejo de outliers en la columna precio, miramos que no hay precios mayores a 999999
outliers_precio = taller[taller["precio"] > 999999]
display(outliers_precio)

,cliente,producto,precio,cantidad,pais,metodo_pago,fecha


In [20]:
#eliminar outlieres ya que sabemos que no hay precios mayores a 999999 y que el valor especifico fue identificado
taller = taller[taller["precio"] != 999999]

Inicialmente la columna cantidad y fecha tenían un formato incorrecto que no permitía operaciones matemáticas y se les dio el formato adecuado según su etiqueta. Identificamos valores nulos en las columnas precio y cantidad que fueron reemplazados con la mediana para evitar discrepancias en análisis posteriores. En la columna país identificamos y corregimos errores tipográficos y en la columna método de pago transformamos todos los valores a minúsculas al no haber valores nulos o errores tipográficos.

Identificamos alrededor de 20 valores atípicos en la columna precio eliminados ya que al representar un aproximado del 0.4% de los datos no afectaria el analisis posterior.

Analisis con Pandas

In [21]:
taller["total"] = taller["cantidad"] * taller["precio"]


In [22]:
#total vendido
total_vendido = taller["total"].sum()
print(f"Total vendido: {total_vendido}")

Total vendido: 25131342.0


In [23]:
#promedio de ventas
promedio_ventas = taller["total"].median()
print(f"Promedio de ventas: {promedio_ventas.round(2)}")

Promedio de ventas: 3940.0


In [24]:
#venta maxima
venta_maxima = taller["total"].max()
print(f"Venta maxima: {venta_maxima}")
#venta minima
venta_minima = taller["total"].min()  
print(f"Venta minima: {venta_minima}")  

Venta maxima: 17955.0
Venta minima: 10.0


In [25]:
#5 productos con mayor valor vendido
ventas_por_producto = taller.groupby("producto")["total"].sum().sort_values(ascending=False)
display(ventas_por_producto.head(5))

producto
mouse      5400604.0
laptop     5145656.0
monitor    4950937.0
celular    4889434.0
teclado    4744711.0
Name: total, dtype: float64

In [26]:
# Ver el precio promedio real de cada producto
print(taller.groupby("producto")["precio"].median())

producto
celular    1045.0
laptop     1028.0
monitor     998.0
mouse      1041.0
teclado     997.0
Name: precio, dtype: float64


In [27]:
#pais con mas ventas
ventas_por_pais = taller.groupby("pais")["total"].sum().sort_values(ascending=False)
print(ventas_por_pais)

pais
colombia    10867739.0
chile        7233218.0
peru         7030385.0
Name: total, dtype: float64


Analisis con Numpy 

In [28]:
#convertimos las columnas numericas a un array con numpy
datos_1 = taller[["precio", "cantidad"]].to_numpy()

In [29]:
datos_1

array([[1.326e+03, 5.000e+00],
       [5.500e+01, 2.000e+00],
       [1.203e+03, 9.000e+00],
       ...,
       [1.776e+03, 2.000e+00],
       [1.366e+03, 8.000e+00],
       [4.060e+02, 1.000e+00]], shape=(4980, 2))

In [30]:
#Separamos las columnas
precios = datos_1[:, 0]
cantidades = datos_1[:, 1]
#aplicamos vectorizacion para calcular el total vendido
Total = precios * cantidades


In [31]:
#suma total de ventas
print("Total vendido:", np.sum(Total))

Total vendido: 25131342.0


In [32]:
#promedio de ventas
print("Promedio ventas:", np.median(Total))

Promedio ventas: 3940.0


In [33]:
#venta maxima con numpy
print("Venta máxima:", np.max(Total))

Venta máxima: 17955.0


In [34]:
#cantidad de ventas mayores a 1000
print("Ventas mayores a 1000: ", len(Total[Total > 1000]))

Ventas mayores a 1000:  4236


In [35]:
taller["total"].describe()

count     4980.000000
mean      5046.454217
std       4105.846364
min         10.000000
25%       1675.250000
50%       3940.000000
75%       7516.250000
max      17955.000000
Name: total, dtype: float64

In [36]:
taller["total"].median()

np.float64(3940.0)

Una de las ventajas que observo al trabajar con numpy es la simplificacion del codigo, que ofrece el mismo resultado que Pandas, solo que con menor cantidad de codigo y mayor eficiencia en las operaciones matematicas trabajando con arrays.

La vectorizacion es un proceso que hace o aplica una operacion a todos los elementos de un arreglo en lugar de recorrerlos en bucle. Lo que por ende lo hace mas rapido y eficiente. 

La expresion data[:,0] es una forma de seleccionar un arreglo en Numpy.

Interpretación de resultados.

Luego de una correccion a los promedios de venta usando median en lugar de mean los resultados tuvieron mas sentido, Pandas y NumPy arrojan el mismo total y promedio lo que indica consistencia en ambas implementaciones. Se detectaron valores sospechosos en la columna de precio, primeramente intente reemplazarlo con la media, sin embargo arrojaba resultados anomalos por lo que decidí eliminar los valores de 999999. En cuanto a la media  de 5046 no representa los datos correctamente ya que esta inflada por las transacciones de alto valor, por lo que lo mas ideal fue representar los datos con la mediana de 3940. Si estos datos fueran reales tomaria las siguientes decisiones: 

1. Segmentar por metodo de pago: cruzandolo con pais o producto podria revelar oportunidades de negocio.
2. Investigar el origen de los valores atipicos (999999)
3. Analizar la tendencia, agregando ventas por mes revelaria las tendencias criticas para el negocio.